# Анализ факторов конверсии в интернет-магазине

Цель: определить поведенческие и сезонные факторы, связанные с покупкой, и подготовить рекомендации для бизнеса.

In [ ]:
import io
import zipfile
from urllib.request import urlopen

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DATA_URL = 'https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip'

with urlopen(DATA_URL) as response:
    archive = zipfile.ZipFile(io.BytesIO(response.read()))
    with archive.open('online_shoppers_intention.csv') as source:
        df = pd.read_csv(source)

df.head()

## Обзор данных

In [ ]:
print(f'Количество сессий: {len(df):,}')
print(f'Общая конверсия: {df["Revenue"].mean():.1%}')
df.info()

## Конверсия по типу посетителя

In [ ]:
conversion_by_visitor = (
    df.groupby('VisitorType', as_index=False)
    .agg(Количество_сессий=('Revenue', 'size'), Конверсия=('Revenue', 'mean'))
    .rename(columns={'VisitorType': 'Тип посетителя'})
)
conversion_by_visitor

## Сезонность

In [ ]:
month_order = ['Feb', 'Mar', 'May', 'June', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly = df.groupby('Month', as_index=False)['Revenue'].mean()
monthly['Month'] = pd.Categorical(monthly['Month'], categories=month_order, ordered=True)
monthly = monthly.sort_values('Month')

sns.set_theme(style='whitegrid')
plt.figure(figsize=(10, 5))
sns.barplot(data=monthly, x='Month', y='Revenue', color='#4C78A8')
plt.title('Конверсия в покупку по месяцам')
plt.xlabel('Месяц')
plt.ylabel('Конверсия')
plt.ylim(0, 0.3)
plt.show()

## Поведение до покупки

In [ ]:
behavior = df.groupby('Revenue')[['PageValues', 'BounceRates', 'ExitRates']].mean().round(3)
behavior.index = behavior.index.map({False: 'Без покупки', True: 'С покупкой'})
behavior

## Выводы и рекомендации

- Усилить маркетинг перед ноябрьским пиком спроса.
- Проверить воронку возвращающихся посетителей: они составляют основную часть трафика, но конвертируются слабее новых.
- Использовать элементы страниц с высоким PageValues в посадочных страницах.
- Отдельно тестировать офферы в выходные: конверсия там выше.